In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from common_sql import update_table, create_count_table_distinct, create_left_join_table

In [2]:
con = sqlite3.connect("../../../vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "../../../v33.db" AS v33')

In [8]:
# define table names 

trans_head = "transaction_head"
transactions = "transaction_v2"
transactions_kohakaandes = "transactions_verbs_obl_kohakaandes_wcomps"
transactions_kohakaandes_counts = "transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1"

# 1. tabeli count tabelid
t1 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl1"
t2 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl2"
t3 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl3"
t4 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl4"
t5 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl5"
t6 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl6"
t7 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl7"

# temp join tables
j1 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j1"
j2 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j2"
j3 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j3"
j4 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j4"
j5 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j5"
j6 = "transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j6"

# 2. tabeli count tabelid
verb_case_counts1 = "transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1"
verb_case_counts2 = "transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2"
verb_case_counts3 = "transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3"
verb_case_counts_temp = "transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1"
verb_case_counts = "transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1"

## 1. tabel 
## verb -> palju esineb obl+kääne (6 kohakäänet) : mitu distinct root 

Võtta välja kõik mis on kohakäändes ja sõna deprel on obl

In [187]:
%%time

cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = transactions_kohakaandes))

cur.execute("""
Create table {new_table} as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tbl1.verb_compound as verb_compound,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM 
{head_table} as tbl1
join 
{trans_table} as tr
on tbl1.id = tr.head_id
where tr.deprel = 'obl'
and 
(INSTR(',' || tr.feats || ',', ',' || 'abl' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'adit' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'all' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ad' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'el' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ill' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'in' || ',') > 0
)
""".format(new_table = transactions_kohakaandes,head_table=trans_head, trans_table=transactions))

CPU times: user 16.5 s, sys: 4.11 s, total: 20.6 s
Wall time: 34.7 s


#### alustabelisse juurde veergu 'kaane', mis käändega on tegu

In [188]:
cur.execute("""ALTER TABLE {tbl} ADD kaane VARCHAR(50)""".format(tbl=transactions_kohakaandes))
con.commit()

update_table(con, transactions_kohakaandes, "kaane", "'abl'", "INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'adit'", "INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'all'", "INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'ad'", "INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'el'", "INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'ill'", "INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0")
update_table(con, transactions_kohakaandes, "kaane", "'in'", "INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0")


#### vaade mis on tabelis

In [189]:
tables = cur.execute("""SELECT * FROM {tbl} limit 10""".format(tbl=transactions_kohakaandes))

for i, elem in enumerate(tables):
    print(elem)

(2, 'toimuma', '', 1, 'lõpp', 'obl', 'S', 'com,in,sg', 'UNK', 'UNK', 'in')
(3, 'saama', 'pihta', 7, 'keel', 'obl', 'S', 'all,com,pl', 'UNK', 'UNK', 'all')
(10, 'tulema', '', 19, 'sina', 'obl', 'P', 'ad,sg', 'UNK', 'YES', 'ad')
(11, 'viilima', '', 22, 'tund', 'obl', 'S', 'com,el,pl', 'UNK', 'UNK', 'el')
(11, 'viilima', '', 23, 'juht', 'obl', 'S', 'ad,com,sg', 'UNK', 'YES', 'ad')
(25, 'muutuma', '', 40, 'mis', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(33, 'minema', 'peale', 59, 'rahvas', 'obl', 'S', 'all,com,sg', 'UNK', 'UNK', 'all')
(37, 'tekkima', '', 69, 'see', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(51, 'kutsuma', '', 85, 'elu', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')
(53, 'tulema', '', 88, 'toim', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')


### base tabel, kus on distinct verbid eelmisest tabelist ja iga kohakäände jaoks count veerg 

### distinct root count

In [31]:
%%time

create_count_table_distinct(con, transactions_kohakaandes, t1, ["verb","verb_compound"], "root_word", "abl_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t2, ["verb","verb_compound"], "root_word", "adit_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t3, ["verb","verb_compound"], "root_word", "all_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t4, ["verb","verb_compound"], "root_word", "ad_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t5, ["verb","verb_compound"], "root_word", "el_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t6, ["verb","verb_compound"], "root_word", "ill_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0", ["verb", "verb_compound"])

create_count_table_distinct(con, transactions_kohakaandes, t7, ["verb","verb_compound"], "root_word", "in_cnt", 
                  "INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0", ["verb", "verb_compound"])

CPU times: user 13.5 s, sys: 954 ms, total: 14.5 s
Wall time: 14.5 s


In [32]:
%%time

create_left_join_table(con, source_tbl1=t1, source_tbl2=t2,result_table=j1,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j1, source_tbl2=t3,result_table=j2,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j2, source_tbl2=t4, result_table=j3,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j3, source_tbl2=t5,result_table=j4,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j4, source_tbl2=t6, result_table=j5,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt", "ill_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j5, source_tbl2=t7, result_table=j6,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt", "ill_cnt", "in_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

CPU times: user 63.9 ms, sys: 5.98 ms, total: 69.9 ms
Wall time: 103 ms


In [41]:
%%time

cur.execute("""
CREATE TABLE {count_tbl} AS select * from {join_tbl}
""".format(count_tbl=transactions_kohakaandes_counts, join_tbl=j6))

update_table(con, transactions_kohakaandes_counts, "abl_cnt", 0, "abl_cnt is null")
update_table(con, transactions_kohakaandes_counts, "adit_cnt",0, "adit_cnt is null")
update_table(con, transactions_kohakaandes_counts, "all_cnt", 0, "all_cnt is null")
update_table(con, transactions_kohakaandes_counts, "ad_cnt", 0, "ad_cnt is null")
update_table(con, transactions_kohakaandes_counts, "el_cnt", 0, "el_cnt is null")
update_table(con, transactions_kohakaandes_counts, "ill_cnt", 0, "ill_cnt is null")
update_table(con, transactions_kohakaandes_counts, "in_cnt", 0, "in_cnt is null")


CPU times: user 7.06 ms, sys: 6.03 ms, total: 13.1 ms
Wall time: 34.5 ms


In [ ]:
for tbl in [j1, j2, j3, j4, j5, j6]:
    cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = tbl))

In [42]:
query = """SELECT * from {tbl}""".format(tbl=transactions_kohakaandes_counts)
source = pd.read_sql_query(query, con)
source

,verb,verb_compound,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,12olema,,1,0,0,0,0,0,0
1,A. tihkama,,1,0,0,0,0,0,0
2,J. teppima,,1,0,0,0,0,0,0
3,Liitootama,,1,0,0,1,0,0,0
4,Southolema,,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
5651,ütlema,ära,5,2,138,67,488,3,54
5652,üürima,,69,18,107,67,49,9,153
5653,üürima,kokku,1,0,0,0,0,0,0
5654,šantažeerima,välja,1,0,0,0,0,0,0


## 2. tabel

### iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [9]:
%%time

cur.execute("""DROP table if exists {tbl}""".format(tbl=verb_case_counts1))

cur.execute("""
CREATE TABLE {new_table} AS
select verb, verb_compound, kaane, count(distinct root_word) as root_cnt
from {verbtable}
group by verb, verb_compound, kaane
""".format(new_table=verb_case_counts1, verbtable=transactions_kohakaandes))

CPU times: user 21 s, sys: 671 ms, total: 21.6 s
Wall time: 21.7 s


In [10]:
%%time

cur.execute("""DROP table if exists {tbl}""".format(tbl=verb_case_counts2))

cur.execute("""
CREATE TABLE {new_table} AS
select  verb, verb_compound, kaane, count(distinct root_word) as elus_cnt
from {verbtable}
where elus='YES'
group by verb, verb_compound, kaane
""".format(new_table=verb_case_counts2,verbtable=transactions_kohakaandes))

CPU times: user 2.9 s, sys: 302 ms, total: 3.2 s
Wall time: 3.22 s


In [14]:
%%time

cur.execute("""DROP table if exists {tbl}""".format(tbl=verb_case_counts3))

cur.execute("""
CREATE TABLE {new_table} AS
select  verb, verb_compound, kaane, count(distinct root_word) as koht_cnt
from {verbtable}
where koht='YES'
group by verb, verb_compound, kaane
""".format(new_table=verb_case_counts3,verbtable=transactions_kohakaandes))

CPU times: user 2.18 s, sys: 273 ms, total: 2.46 s
Wall time: 2.47 s


In [15]:
create_left_join_table(con, source_tbl1=verb_case_counts1, source_tbl2=verb_case_counts2, result_table=verb_case_counts_temp,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "tbl1.kaane", "elus_cnt", "root_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound and tbl1.kaane = tbl2.kaane")

create_left_join_table(con, source_tbl1=verb_case_counts_temp, source_tbl2=verb_case_counts3, result_table=verb_case_counts,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "tbl1.kaane", "tbl1.elus_cnt","koht_cnt", "tbl1.root_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound and tbl1.kaane = tbl2.kaane")


In [16]:
update_table(con, verb_case_counts, "elus_cnt", 0, "elus_cnt is null")
update_table(con, verb_case_counts, "koht_cnt", 0, "koht_cnt is null")

In [19]:
query = """select * from {tbl} order by root_cnt desc""".format(tbl = verb_case_counts)
source = pd.read_sql_query(query, con)
source

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,saama,,el,1274,402,19632
1,andma,,all,1601,250,11612
2,rääkima,,el,802,202,10891
3,saama,,in,134,306,8532
4,tulema,,ad,1134,166,8468
...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1
74716,švipsima,,ad,0,0,1
74717,žestikuleerima,,ad,0,1,1
74718,žisraelima,,ad,0,0,1


In [45]:
source.to_csv(verb_case_counts+'.csv', index=False, sep=",", encoding="utf-8")

In [20]:
con.close()